In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [3]:
base_url = "https://imsdb.com/"
start_url = "https://imsdb.com/all-scripts.html"

In [4]:
# gets the page with all script links
response = requests.get(start_url)
soup = BeautifulSoup(response.text, "html.parser")

# find links to movie pages
movie_links = []

for a in soup.select("p a"):
    href = a.get("href")
    if href and href.startswith("/Movie Scripts/"):
        movie_links.append(urljoin(base_url, href))

print("Movie pages found:", len(movie_links))
print(movie_links[:10])


Movie pages found: 1297
['https://imsdb.com/Movie Scripts/10 Things I Hate About You Script.html', 'https://imsdb.com/Movie Scripts/12 Script.html', 'https://imsdb.com/Movie Scripts/12 and Holding Script.html', 'https://imsdb.com/Movie Scripts/12 Monkeys Script.html', 'https://imsdb.com/Movie Scripts/12 Years a Slave Script.html', 'https://imsdb.com/Movie Scripts/127 Hours Script.html', 'https://imsdb.com/Movie Scripts/1492: Conquest of Paradise Script.html', 'https://imsdb.com/Movie Scripts/15 Minutes Script.html', 'https://imsdb.com/Movie Scripts/17 Again Script.html', 'https://imsdb.com/Movie Scripts/187 Script.html']


In [ ]:
data = []

for movie_url in movie_links[:20]:   # first 20 movies

    movie_page = requests.get(movie_url)
    movie_soup = BeautifulSoup(movie_page.text, "html.parser")

    # get title
    title = movie_soup.title.get_text(strip=True).split(" Script")[0]

    # find script link
    script_link = None
    for a in movie_soup.find_all("a"):
        if "Read" in a.get_text():
            script_link = urljoin(base_url, a.get("href"))
            break

    # open script page
    script_page = requests.get(script_link)
    script_soup = BeautifulSoup(script_page.text, "html.parser")

    script_container = script_soup.find("td", class_="scrtext")

    if script_container:
        script_text = script_container.get_text("\n", strip=True)

        data.append({
            "title": title,
            "script": script_text
        })



df = pd.DataFrame(data)
print(df.head())

                        title  \
0  10 Things I Hate About You   
1                          12   
2              12 and Holding   
3                  12 Monkeys   
4            12 Years a Slave   

                                              script  
0  Ten Things I Hate About You - by Karen McCulla...  
1  12 - Script\nCUT FROM BLACK\nTITLE:\nFIN\nEXTE...  
2  12 AND HOLDING\nWritten by\nAnthony S Cipriano...  
3  Twelve Monkeys\nTWELVE MONKEYS\nAn original sc...  
4  12 YEARS A SLAVE\nWritten by\r\n\r\n          ...  


In [ ]:
import re
import numpy as np
import pandas as pd


MANUAL_GENRE_VECTORS = {
    "Horror": [
        "blood", "scream", "killer", "dark", "death", "fear", "ghost", "knife", "night", "evil",
        "dead", "body", "monster", "shadow", "haunted", "attack", "hide", "danger", "panic", "cry",
        "terrified", "grave", "curse", "demon", "victim", "murder", "skull", "forest", "shock", "creature"
    ],
    "Crime": [
        "police", "money", "gun", "car", "bank", "street", "boss", "deal", "stole", "detective",
        "robbery", "murder", "case", "criminal", "evidence", "cop", "mafia", "hit", "jail", "drug",
        "suspect", "thief", "lawyer", "court", "gang", "escape", "witness", "track", "investigate", "bullet"
    ],
    "War": [
        "army", "battle", "soldier", "enemy", "captain", "general", "attack", "bomb", "fight", "weapon",
        "mission", "troop", "tank", "warfare", "gunfire", "uniform", "commander", "base", "front", "march",
        "victory", "defeat", "radio", "orders", "camp", "battlefield", "explosion", "rifle", "navy", "aircraft"
    ],
    "Drama": ["emotion", "emotional", "crying", "fight", "fighting", "betrayal", "argument", "regret", "love",
        "cheat", "cheating", "pain", "loss", "grief", "nostalgia", "frustrated", "confused", "miscalculations", 
        "misunderstanding", "significant", "partner", "decision", "choice", "regret", "memory", "memories", "sad",
        "life", "change", "changes", "trust", "lies", "lying", "conflict", "bond", "bonded"
    ],
    "Thriller": ["murder", "crime", "secret", "suspense", "dangerous", "killer", "victim", "traitor", "spy", "spies",
        "agent", "weapon", "weapons", "gun", "guns", "investigation", "hidden", "suspense", "tense", "trap", 
        "escape", "night", "dark", "revenge", "observe", "observed", "evidence", "clue", "betrayal", "panic",
        "discover", "hide", "hiding", "sudden", "suddenly"
    ]
    
}

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "had", "has", "have",
    "he", "her", "hers", "him", "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "she", "that", "the", "their", "them", "they", "this", "to", "was",
    "we", "were", "with", "you", "your"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [word for word in tokens if word not in STOP_WORDS and len(word) > 1]

def read_text_file(file_name):
    with open(file_name, "r", encoding="utf-8") as f:
        return f.read()

def count_matches(tokens, genre_words):
    counts = pd.Series(tokens).value_counts()
    return sum(counts.get(word, 0) for word in genre_words)

def cosine_similarity(vec1, vec2):
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    if denom == 0:
        return 0.0
    return float(np.dot(vec1, vec2) / denom)

for movie in data:

    title = movie["title"]
    script_text = movie["script"]

    tokens = clean_text(script_text)

    vocabulary = []
    for words in MANUAL_GENRE_VECTORS.values():
        for word in words:
            if word not in vocabulary:
                vocabulary.append(word)

    script_vector = np.array([tokens.count(word) for word in vocabulary], dtype=float)

    scores = {}
    for genre, words in MANUAL_GENRE_VECTORS.items():
        genre_vector = np.array([1 if word in words else 0 for word in vocabulary], dtype=float)
        scores[genre] = cosine_similarity(script_vector, genre_vector)

    print("\n-------------------------")
    print("Movie:", title)

    print("Similarity scores:")
    for genre, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(genre, round(score, 4))

    print("\nRaw matching word counts:")
    for genre, words in MANUAL_GENRE_VECTORS.items():
        print(genre, count_matches(tokens, words))

    predicted_genre = max(scores, key=scores.get)
    print("\nPredicted genre:", predicted_genre)


-------------------------
Movie: 10 Things I Hate About You
Similarity scores:
Horror 0.2257
Thriller 0.1821
Crime 0.1649
Drama 0.1098
War 0.055

Raw matching word counts:
Horror 78
Crime 57
War 19
Drama 41
Thriller 67

Predicted genre: Horror

-------------------------
Movie: 12
Similarity scores:
Horror 0.2025
Crime 0.198
Thriller 0.1672
War 0.109
Drama 0.0577

Raw matching word counts:
Horror 91
Crime 89
War 49
Drama 28
Thriller 80

Predicted genre: Horror

-------------------------
Movie: 12 and Holding
Similarity scores:
Horror 0.2667
Drama 0.2313
Thriller 0.2294
Crime 0.174
War 0.1011

Raw matching word counts:
Horror 95
Crime 62
War 36
Drama 89
Thriller 87

Predicted genre: Horror

-------------------------
Movie: 12 Monkeys
Similarity scores:
Thriller 0.3433
Crime 0.2698
Horror 0.2258
War 0.1584
Drama 0.0973

Raw matching word counts:
Horror 144
Crime 172
War 101
Drama 67
Thriller 233

Predicted genre: Thriller

-------------------------
Movie: 12 Years a Slave
Similarity scor

In [36]:
# testing individual movies
movie_name = "10 Things I Hate About You"

for movie in data:
    if movie["title"].lower() == movie_name.lower():

        tokens = clean_text(movie["script"])

        script_vector = np.array([tokens.count(word) for word in vocabulary], dtype=float)

        scores = {}
        for genre, words in MANUAL_GENRE_VECTORS.items():
            genre_vector = np.array([1 if word in words else 0 for word in vocabulary], dtype=float)
            scores[genre] = cosine_similarity(script_vector, genre_vector)

        print("Movie:", movie["title"])

    print("Similarity scores:")
    for genre, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
        print(genre, round(score, 4))

    print("\nRaw matching word counts:")
    for genre, words in MANUAL_GENRE_VECTORS.items():
        print(genre, count_matches(tokens, words))

    predicted_genre = max(scores, key=scores.get)
    print("\nPredicted genre:", predicted_genre)
    break

Movie: 10 Things I Hate About You
Similarity scores:
Horror 0.2257
Thriller 0.1821
Crime 0.1649
Drama 0.1098
War 0.055

Raw matching word counts:
Horror 78
Crime 57
War 19
Drama 41
Thriller 67

Predicted genre: Horror
